# Web Scraping - Wikipedia
Atividade de Fundamentos e Técnicas em Ciência de Dados - UFRN.


In [ ]:
!pip -q install requests beautifulsoup4 scrapy crochet wordcloud matplotlib

In [ ]:
import re
import time
from urllib.parse import quote
import requests
from bs4 import BeautifulSoup
from wordcloud import WordCloud
import matplotlib.pyplot as plt

STOPWORDS_PT = {'a','ao','aos','as','com','como','da','das','de','do','dos','e','ela','ele','em','entre','essa','esse','esta','este','foi','foram','há','isso','isto','já','mais','mas','na','nas','não','no','nos','o','os','ou','para','pela','pelo','por','que','quem','se','sem','ser','sua','suas','seu','seus','também','tem','ter','um','uma','uns','umas'}

def montar_url(termo):
    termo = termo.strip().replace(' ', '_')
    return 'https://pt.wikipedia.org/wiki/' + quote(termo, safe='_()-')

def limpar_texto(texto):
    palavras = re.findall(r'[a-záàâãéêíóôõúüç]+', texto.lower())
    palavras = [p for p in palavras if p not in STOPWORDS_PT and len(p) > 2]
    return ' '.join(palavras)

def mostrar_nuvem(texto):
    nuvem = WordCloud(width=1000, height=500, background_color='white', collocations=False).generate(texto)
    plt.figure(figsize=(12,6))
    plt.imshow(nuvem, interpolation='bilinear')
    plt.axis('off')
    plt.show()

## 1) Requests + BeautifulSoup

In [ ]:
def scraping_requests(termos):
    inicio = time.perf_counter()
    textos = []
    for termo in termos:
        url = montar_url(termo)
        resposta = requests.get(url, headers={'User-Agent':'UFRN-Atividade/1.0'}, timeout=20)
        resposta.raise_for_status()
        soup = BeautifulSoup(resposta.text, 'html.parser')
        paragrafos = soup.select('div.mw-parser-output p') or soup.find_all('p')
        textos.append(' '.join(p.get_text(' ', strip=True) for p in paragrafos))
    tempo = time.perf_counter() - inicio
    return ' '.join(textos), tempo

entrada = input('Digite 5 termos separados por vírgula: ')
termos = [t.strip() for t in entrada.split(',') if t.strip()]

if len(termos) != 5:
    print('Digite exatamente 5 termos.')
else:
    texto_requests, tempo_requests = scraping_requests(termos)
    texto_requests_limpo = limpar_texto(texto_requests)
    print(f'Tempo Requests + BeautifulSoup: {tempo_requests:.2f} segundos')
    mostrar_nuvem(texto_requests_limpo)
    palavra = input('Digite uma palavra para contar: ').lower().strip()
    print('Quantidade:', texto_requests_limpo.split().count(palavra))

## 2) Scrapy + Crochet no Colab

In [ ]:
import scrapy
from scrapy.crawler import CrawlerRunner
from crochet import setup, wait_for

setup()
resultados_scrapy = []

class WikipediaSpider(scrapy.Spider):
    name = 'wikipedia_colab'
    custom_settings = {'LOG_ENABLED': False, 'USER_AGENT': 'UFRN-Atividade/1.0'}

    def __init__(self, urls=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.start_urls = urls or []

    def parse(self, response):
        partes = response.css('div.mw-parser-output p ::text').getall()
        resultados_scrapy.append(' '.join(p.strip() for p in partes if p.strip()))

runner = CrawlerRunner()

@wait_for(timeout=120)
def executar_scrapy(urls):
    return runner.crawl(WikipediaSpider, urls=urls)

entrada = input('Digite 5 termos separados por vírgula: ')
termos = [t.strip() for t in entrada.split(',') if t.strip()]

if len(termos) != 5:
    print('Digite exatamente 5 termos.')
else:
    resultados_scrapy.clear()
    urls = [montar_url(t) for t in termos]
    inicio = time.perf_counter()
    executar_scrapy(urls)
    tempo_scrapy = time.perf_counter() - inicio
    texto_scrapy = ' '.join(resultados_scrapy)
    texto_scrapy_limpo = limpar_texto(texto_scrapy)
    print(f'Tempo Scrapy: {tempo_scrapy:.2f} segundos')
    mostrar_nuvem(texto_scrapy_limpo)
    palavra = input('Digite uma palavra para contar: ').lower().strip()
    print('Quantidade:', texto_scrapy_limpo.split().count(palavra))